# Meow Clinic — Cat Care Assistant

**Haifa Ahmed Alsulami · Wed Ayad Alshalawi**

Building Agentic AI Systems — SDAIA Academy, 9–13 August 2026
Track C · complete

---

An assistant for the clients of a cat clinic. Questions fall into two very
different kinds — *"my cat is scratching the sofa"* and *"how much is a
check-up"* — so the assistant keeps two separate knowledge bases and picks one
per question:

- **Care guide** — feeding, behaviour, grooming, breeds, warning signs
- **Clinic services** — prices, booking rules, vaccination schedule, boarding

Later parts add the routing classifier, memory across conversations, and a
human approval step before booking.

*This is a training project. It does not diagnose illness and is not a
substitute for a veterinarian.*

**Part 1:** setup, the two knowledge bases, retrieval, calculation tools.
**Part 2:** the routing classifier, and memory that survives across
conversations.
**Part 3:** the same flow rebuilt with the functional API, with retries,
fallbacks, and a human approval step before booking.
**Part 4:** the evaluator-optimizer loop, and tracing in LangSmith.

## Setup

In [1]:
%pip install -qU langchain langchain-groq langgraph langchain-community \
    langchain-huggingface sentence-transformers chromadb langsmith

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.0/147.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.9/248.9 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 611.3/611.3 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 735.3/735.3 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 565.1/565.1 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2

In [2]:
import os

# keep model-download progress bars out of the saved notebook
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TQDM_DISABLE"] = "1"
os.environ["CHROMA_SERVER_NO_TELEMETRY"] = "1"

# LangSmith. The variable is LANGCHAIN_TRACING_V2 — LANGSMITH_TRACING_V2 is
# not a real variable and fails silently. It has to be set here, before
# langchain is imported, or the tracer is built without it and nothing is
# ever sent.
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "meow-clinic-capstone"

try:
    from google.colab import userdata
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
    os.environ["LANGCHAIN_API_KEY"] = userdata.get("LANGCHAIN_API_KEY")
    print("keys loaded from Colab secrets")
except Exception as e:
    import getpass
    os.environ["GROQ_API_KEY"] = getpass.getpass("GROQ_API_KEY: ")
    os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LANGCHAIN_API_KEY: ")
    print("keys entered manually")

print("groq     :", bool(os.environ.get("GROQ_API_KEY")))
print("langsmith:", bool(os.environ.get("LANGCHAIN_API_KEY")))
print("tracing  :", os.environ.get("LANGCHAIN_TRACING_V2"))

keys loaded from Colab secrets
groq     : True
langsmith: True
tracing  : true


Groq retires model names periodically, so we try a few and keep the first one
that answers.

In [3]:
from langchain_groq import ChatGroq

CANDIDATES = [
    "llama-3.3-70b-versatile",
    "llama-3.1-8b-instant",
    "openai/gpt-oss-120b",
    "moonshotai/kimi-k2-instruct",
]

llm = None
for name in CANDIDATES:
    try:
        candidate = ChatGroq(model=name, temperature=0)
        candidate.invoke("hi")
        llm = candidate
        MODEL_NAME = name
        print(f"using: {name}")
        break
    except Exception as e:
        print(f"skip {name}: {type(e).__name__} — {str(e)[:110]}")

if llm is None:
    raise RuntimeError(
        "No model worked. Check the key, then run:\n"
        "  import requests, os\n"
        "  r = requests.get('https://api.groq.com/openai/v1/models',\n"
        "      headers={'Authorization': 'Bearer ' + os.environ['GROQ_API_KEY']})\n"
        "  print([m['id'] for m in r.json()['data']])"
    )

using: llama-3.3-70b-versatile


## Knowledge bases

Two documents, written so that neither can answer the other's questions: the
care guide has no prices or procedures, and the services guide has no care
advice. Both are written to disk here so the notebook runs without any
uploads.

In [4]:
care_guide = """
# دليل رعاية القطط — عيادة مواء

## التغذية اليومية
تحتاج القطة البالغة وجبتين إلى ثلاث وجبات موزعة على اليوم. القطط آكلة لحوم
إجبارية، ويجب أن يكون البروتين الحيواني المكون الأول في الطعام. الطعام الرطب
مهم لأنه يرفع كمية الماء الداخلة للجسم، والقطط بطبيعتها ضعيفة الإحساس بالعطش.
تجنب إعطاء القطة الحليب البقري، فمعظم القطط البالغة لا تهضم اللاكتوز وتصاب
بالإسهال.

## الماء
وفر ماءً نظيفاً متجدداً في أكثر من مكان في المنزل. كثير من القطط تفضل الماء
المتحرك، ونافورة الشرب تزيد كمية ما تشربه بشكل ملحوظ. ضع وعاء الماء بعيداً عن
وعاء الطعام وبعيداً عن صندوق الرمل.

## صندوق الرمل
القاعدة صندوق لكل قطة وزيادة صندوق واحد. نظف الصندوق يومياً وغيّر الرمل كاملاً
أسبوعياً. امتناع القطة عن استخدام الصندوق فجأة ليس عناداً، بل إشارة إلى مشكلة
صحية أو توتر في البيئة، ويستدعي الانتباه.

## الخربشة وتخريب الأثاث
الخربشة سلوك طبيعي لشحذ الأظافر وترك علامة رائحة، ولا يمكن منعها بالعقاب. ضع
عمود خربشة طويلاً بما يكفي لتمدد القطة كاملة، وثبته بقرب المكان الذي تخربشه
حالياً. قص أطراف الأظافر كل أسبوعين إلى ثلاثة أسابيع يقلل الضرر.

## العناية بالفرو
السلالات طويلة الشعر مثل الشيرازي والهيمالايا تحتاج تمشيطاً يومياً، وإهمال ذلك
يؤدي إلى تكتلات مؤلمة تلتصق بالجلد. السلالات قصيرة الشعر يكفيها التمشيط مرتين
أسبوعياً. التمشيط المنتظم يقلل كرات الشعر التي تبتلعها القطة.

## القطط الجديدة في المنزل
عند وصول قطة جديدة، اعزلها في غرفة واحدة هادئة فيها كل احتياجاتها لمدة أسبوع
تقريباً، ثم وسّع نطاقها تدريجياً. التعريف بين قطتين يتم بالرائحة أولاً عبر
تبادل الأغطية، ثم الرؤية من خلف حاجز، ثم اللقاء المباشر القصير.

## سلالة الشيرازي
وجه مسطح وفرو طويل كثيف. عرضة لمشكلات التنفس بسبب قصر الأنف، ولإفرازات العين
التي تحتاج مسحاً يومياً بقطعة قطنية مبللة. تحتاج تمشيطاً يومياً وحمّاماً كل
أربعة إلى ستة أسابيع.

## سلالة السيامي
سلالة نشطة اجتماعية عالية الصوت، تحتاج تفاعلاً ولعباً يومياً وتعاني من الملل
والوحدة إذا تركت وحدها طويلاً. فروها قصير ويكفيها تمشيط أسبوعي.

## علامات تستدعي عرض القطة على طبيب فوراً
امتناع القطة عن الأكل لأكثر من أربع وعشرين ساعة. محاولة التبول بلا نتيجة أو
التبول المتكرر بكميات قليلة، وهي حالة طارئة في الذكور تحديداً. التنفس بفم
مفتوح أو اللهاث. القيء المتكرر أكثر من مرتين في اليوم. الخمول الشديد وعدم
الاستجابة. نزيف من أي مكان. تورم البطن. عرج مفاجئ أو عدم القدرة على الوقوف.
هذه الحالات لا تحتمل الانتظار ولا تُعالج منزلياً.

## كرات الشعر
ابتلاع الشعر أثناء التنظيف الذاتي أمر طبيعي، وخروج كرة شعر من حين لآخر ليس
مقلقاً. لكن تكرار محاولات الإخراج دون خروج شيء قد يعني انسداداً، وهو ما يحتاج
فحصاً.
"""

with open("care_guide.md", "w", encoding="utf-8") as f:
    f.write(care_guide)

print(len(care_guide), "characters written to care_guide.md")

2430 characters written to care_guide.md


In [5]:
clinic_services = """
# دليل خدمات عيادة مواء

## ساعات العمل
تعمل العيادة من السبت إلى الخميس من التاسعة صباحاً حتى التاسعة مساءً، والجمعة
من الرابعة عصراً حتى التاسعة مساءً. قسم الطوارئ يعمل على مدار الساعة طوال
أيام الأسبوع.

## رسوم الكشف
الكشف العام مئة وخمسون ريالاً. الكشف في قسم الطوارئ خارج ساعات العمل ثلاثمئة
ريال. إعادة الكشف خلال أربعة عشر يوماً من الزيارة الأولى لنفس الحالة مجانية.
استشارة الطبيب عبر الهاتف ثمانون ريالاً.

## أسعار التطعيمات
التطعيم الرباعي مئتان وعشرون ريالاً. تطعيم السعار مئة وثمانون ريالاً. تطعيم
لوكيميا القطط مئتان وخمسون ريالاً. باقة التطعيمات الأساسية الكاملة خمسمئة
وخمسون ريالاً وتشمل الرباعي والسعار وفحصاً عاماً.

## جدول التطعيمات
تبدأ القطة الصغيرة التطعيم الرباعي في عمر ثمانية أسابيع، وتُكرر الجرعة كل ثلاثة
أسابيع حتى عمر ستة عشر أسبوعاً. يُعطى تطعيم السعار في عمر اثني عشر أسبوعاً.
بعد ذلك يُجدد الرباعي والسعار مرة واحدة كل اثني عشر شهراً. تطعيم اللوكيميا
يُعطى للقطط التي تخرج من المنزل أو تخالط قططاً أخرى، ويُجدد سنوياً.

## حجز المواعيد
يتم الحجز عبر تطبيق العيادة أو بالاتصال. يُطلب مبلغ تأكيد قدره خمسون ريالاً
لحجز أي موعد، ويُخصم من قيمة الزيارة. مواعيد الطوارئ لا تحتاج حجزاً مسبقاً.

## الإلغاء والتأخير
الإلغاء قبل الموعد بأربع وعشرين ساعة أو أكثر يُسترد فيه مبلغ التأكيد كاملاً.
الإلغاء خلال أقل من أربع وعشرين ساعة لا يُسترد فيه المبلغ. التأخر عن الموعد
أكثر من خمس عشرة دقيقة يُلغى معه الحجز ويُعاد جدولته حسب التوافر.

## الإيواء
الإيواء متاح للقطط المطعّمة فقط، ويُشترط أن يكون آخر تطعيم ساري المفعول.
الغرفة العادية ثمانون ريالاً في الليلة. الجناح الواسع بنافذة مئة وأربعون
ريالاً في الليلة. مرافقة قطة ثانية من نفس المنزل في الغرفة ذاتها بنصف السعر.
الحجوزات لخمس ليالٍ فأكثر عليها خصم عشرة بالمئة على إجمالي الليالي.

## خدمات إضافية أثناء الإيواء
جلسة لعب يومية إضافية أربعون ريالاً في اليوم. الاستحمام والتنظيف الكامل قبل
الاستلام مئة وعشرون ريالاً. تقديم دواء موصوف ثلاثون ريالاً في اليوم.

## خدمات العناية
تقليم الأظافر أربعون ريالاً. الاستحمام والتجفيف مئة وعشرة ريالات. الحلاقة
الكاملة للسلالات طويلة الشعر مئتان ريال. تنظيف الأسنان تحت التخدير ثمانمئة
ريال ويشمل الفحص قبل التخدير.

## التعقيم والخصي
خصي الذكر خمسمئة وخمسون ريالاً. تعقيم الأنثى ثمانمئة وخمسون ريالاً. يشمل
السعر التخدير والمتابعة بعد العملية والغيار وزيارة المراجعة. يُشترط صيام القطة
عن الطعام اثنتي عشرة ساعة قبل العملية.

## العضوية السنوية
العضوية السنوية ألف ومئتا ريال، وتشمل كشفين عامين، وباقة التطعيمات الأساسية،
وخصم خمسة عشر بالمئة على خدمات العناية والإيواء.
"""

with open("clinic_services.md", "w", encoding="utf-8") as f:
    f.write(clinic_services)

print(len(clinic_services), "characters written to clinic_services.md")

2396 characters written to clinic_services.md


## Indexing

Each document is split on its `##` headings so a section stays in one chunk,
then embedded into its own Chroma collection. Two collections rather than one
filtered collection, so that routing to the wrong source really does fail to
find the answer.

The corpus is Arabic, so the embedding model has to be multilingual —
`all-MiniLM-L6-v2` would retrieve almost at random here.

In [6]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=40,
    separators=["\n## ", "\n\n", "\n", " "],
)

care_docs   = TextLoader("care_guide.md", encoding="utf-8").load()
clinic_docs = TextLoader("clinic_services.md", encoding="utf-8").load()

care_chunks   = splitter.split_documents(care_docs)
clinic_chunks = splitter.split_documents(clinic_docs)

for c in care_chunks:
    c.metadata["source"] = "care_guide"
for c in clinic_chunks:
    c.metadata["source"] = "clinic_services"

print("care   :", len(care_chunks), "chunks")
print("clinic :", len(clinic_chunks), "chunks")
print()
print(care_chunks[0].page_content[:250])

/tmp/ipykernel_3481/3397451603.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


care   : 13 chunks
clinic : 13 chunks

# دليل رعاية القطط — عيادة مواء


In [7]:
from langchain_huggingface import HuggingFaceEmbeddings
import numpy as np

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

def cos(a, b):
    a, b = np.array(a), np.array(b)
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

# quick check that the model handles Arabic: the related pair should score higher
v_scratch = embeddings.embed_query("قطتي تخربش الكنب كيف أوقفها؟")
v_claws   = embeddings.embed_query("سلوك الخربشة وعمود الخربشة")
v_price   = embeddings.embed_query("كم سعر تطعيم السعار؟")

print(f"related   : {cos(v_scratch, v_claws):.3f}")
print(f"unrelated : {cos(v_scratch, v_price):.3f}")

related   : 0.455
unrelated : -0.017


In [8]:
from langchain_community.vectorstores import Chroma

care_store = Chroma.from_documents(
    care_chunks, embeddings, collection_name="care_guide")
clinic_store = Chroma.from_documents(
    clinic_chunks, embeddings, collection_name="clinic_services")

care_retriever   = care_store.as_retriever(search_kwargs={"k": 5})
clinic_retriever = clinic_store.as_retriever(search_kwargs={"k": 5})

print("care   :", care_store._collection.count(), "vectors")
print("clinic :", clinic_store._collection.count(), "vectors")

care   : 13 vectors
clinic : 13 vectors


If `Chroma.from_documents` fails on a fresh Colab runtime, it is a packaging
conflict in Colab rather than an error in this code. Run the two commands
below once, restart the session, and run from the top.

```
!pip uninstall -y chromadb opentelemetry-api opentelemetry-sdk numpy
!pip install -qU chromadb==0.4.18 opentelemetry-api==1.42.1 \
    opentelemetry-sdk==1.42.1 'numpy<2.0.0'
```

### Checking retrieval

Each store is asked its own question, then the other store's question. The
first two should come back with the matching section; the last two should come
back with something unrelated, which is what confirms the stores are actually
separate.

In [9]:
def show(docs, label):
    print(f"\n--- {label} ({len(docs)} chunks) ---")
    if not docs:
        print("EMPTY — nothing retrieved")
    for d in docs:
        text = " ".join(d.page_content.split())
        print(f"  [{d.metadata.get('source')}] {text[:140]}...")

q_care   = "قطتي تخربش الأثاث، كيف أتعامل مع هذا السلوك؟"
q_clinic = "كم سعر تطعيم السعار ومتى يتم تجديده؟"

show(care_retriever.invoke(q_care),     "care store, care question")
show(clinic_retriever.invoke(q_clinic), "clinic store, clinic question")
show(care_retriever.invoke(q_clinic),   "care store, clinic question")
show(clinic_retriever.invoke(q_care),   "clinic store, care question")


--- care store, care question (5 chunks) ---
  [care_guide] ## القطط الجديدة في المنزل عند وصول قطة جديدة، اعزلها في غرفة واحدة هادئة فيها كل احتياجاتها لمدة أسبوع تقريباً، ثم وسّع نطاقها تدريجياً. ال...
  [care_guide] ## الخربشة وتخريب الأثاث الخربشة سلوك طبيعي لشحذ الأظافر وترك علامة رائحة، ولا يمكن منعها بالعقاب. ضع عمود خربشة طويلاً بما يكفي لتمدد القطة...
  [care_guide] ## علامات تستدعي عرض القطة على طبيب فوراً امتناع القطة عن الأكل لأكثر من أربع وعشرين ساعة. محاولة التبول بلا نتيجة أو التبول المتكرر بكميات ...
  [care_guide] ## صندوق الرمل القاعدة صندوق لكل قطة وزيادة صندوق واحد. نظف الصندوق يومياً وغيّر الرمل كاملاً أسبوعياً. امتناع القطة عن استخدام الصندوق فجأة...
  [care_guide] # دليل رعاية القطط — عيادة مواء...

--- clinic store, clinic question (5 chunks) ---
  [clinic_services] ## أسعار التطعيمات التطعيم الرباعي مئتان وعشرون ريالاً. تطعيم السعار مئة وثمانون ريالاً. تطعيم لوكيميا القطط مئتان وخمسون ريالاً. باقة التطع...
  [clinic_services] ## العضوية السنوية العضوية السنوية

### Does the retrieved text contain the answer?

Retrieving *something* is not the same as retrieving the right thing. Each
question below has a known answer sitting in the corpus, so the check is a
plain substring test.

In [10]:
# Each question has a known answer in the corpus, so this is a substring test
# rather than a judgement call.
CHECKS = [
    ("كم سعر تقليم الأظافر؟",            clinic_retriever, "أربعون"),
    ("كم سعر الاستحمام والتجفيف للقطة؟", clinic_retriever, "مئة وعشرة"),
    ("كم تكلفة الحلاقة الكاملة؟",        clinic_retriever, "مئتان"),
    ("كم سعر تطعيم السعار؟",             clinic_retriever, "مئة وثمانون"),
    ("كم سعر الغرفة العادية للإيواء؟",   clinic_retriever, "ثمانون"),
    ("كل كم أمشط فرو القط الشيرازي؟",    care_retriever,   "يومياً"),
    ("متى أعرض قطتي على الطبيب فوراً؟",  care_retriever,   "أربع وعشرين"),
    ("هل أعطي قطتي حليب بقر؟",           care_retriever,   "اللاكتوز"),
]

passed = 0
for question, retriever, expected in CHECKS:
    text = " ".join(d.page_content for d in retriever.invoke(question))
    ok = expected in text
    passed += ok
    print(f"{'PASS' if ok else 'FAIL'}  {expected:12} <- {question}")

print(f"\n{passed}/{len(CHECKS)} answers present in the retrieved context")

PASS  أربعون       <- كم سعر تقليم الأظافر؟
FAIL  مئة وعشرة    <- كم سعر الاستحمام والتجفيف للقطة؟
PASS  مئتان        <- كم تكلفة الحلاقة الكاملة؟
PASS  مئة وثمانون  <- كم سعر تطعيم السعار؟
PASS  ثمانون       <- كم سعر الغرفة العادية للإيواء؟
PASS  يومياً       <- كل كم أمشط فرو القط الشيرازي؟
PASS  أربع وعشرين  <- متى أعرض قطتي على الطبيب فوراً؟
PASS  اللاكتوز     <- هل أعطي قطتي حليب بقر؟

7/8 answers present in the retrieved context


One question fails. Rather than adjusting the test until it passes, here is
what the retriever actually returned for it.

In [11]:
q_failed = "كم سعر الاستحمام والتجفيف للقطة؟"

for d in clinic_retriever.invoke(q_failed):
    print(" ".join(d.page_content.split())[:170], "\n")

## الإيواء الإيواء متاح للقطط المطعّمة فقط، ويُشترط أن يكون آخر تطعيم ساري المفعول. الغرفة العادية ثمانون ريالاً في الليلة. الجناح الواسع بنافذة مئة وأربعون ريالاً في الل 

يُعطى للقطط التي تخرج من المنزل أو تخالط قططاً أخرى، ويُجدد سنوياً. 

## التعقيم والخصي خصي الذكر خمسمئة وخمسون ريالاً. تعقيم الأنثى ثمانمئة وخمسون ريالاً. يشمل السعر التخدير والمتابعة بعد العملية والغيار وزيارة المراجعة. يُشترط صيام القطة  

## خدمات إضافية أثناء الإيواء جلسة لعب يومية إضافية أربعون ريالاً في اليوم. الاستحمام والتنظيف الكامل قبل الاستلام مئة وعشرون ريالاً. تقديم دواء موصوف ثلاثون ريالاً في ال 

## جدول التطعيمات تبدأ القطة الصغيرة التطعيم الرباعي في عمر ثمانية أسابيع، وتُكرر الجرعة كل ثلاثة أسابيع حتى عمر ستة عشر أسبوعاً. يُعطى تطعيم السعار في عمر اثني عشر أسبوع 



The corpus lists bathing twice: `خدمات العناية` charges 110 SAR for
*الاستحمام والتجفيف*, and `خدمات إضافية أثناء الإيواء` charges 120 SAR for
*الاستحمام والتنظيف الكامل قبل الاستلام*. The second wording is closer to the
question, so semantic search ranked it first.

The retriever is behaving correctly on an ambiguity that exists in the source
documents. Fixing it means rewording the two services so they are
distinguishable, not changing the search — and it is left visible here rather
than tidied away.

### Why 2-Step RAG, and not Agentic or Hybrid

**2-Step** — classify once, retrieve once, answer — is what this pipeline does,
and it suits the problem. The two sources are small and cleanly divided, so a
single well-aimed retrieval reaches the answer; there is no second lookup that
depends on the result of the first. The cost profile supports it too: the trace
in Part 4 shows retrieval taking 0.05 s against 0.3–4.2 s for every model call, so
each extra agentic round would add seconds of latency to save a fraction of one.

**Agentic RAG** — letting the model search repeatedly, reformulate, and decide
when it has enough — would earn its cost on questions needing several dependent
lookups. There is exactly one place here where it would help: the ambiguous
bathing question in the check above, where an agent could notice two candidate
prices and ask which service was meant instead of silently picking the nearer
one. That is not enough to justify the latency across all traffic.

**Hybrid** — dense vectors plus keyword search — is the option most worth
revisiting. The corpus is Arabic and the failing question turned on two
near-identical phrasings, which is where BM25 tends to beat embeddings: exact
terms like *الاستحمام والتجفيف* would match literally rather than by
similarity. It was not implemented here because the grounding check passes 7 of
8 without it, and the one failure is a documented ambiguity in the source rather
than a retrieval weakness.

## Tools

Four calculations the assistant can run on a cat's details. The calorie tool
uses the standard resting energy requirement, `RER = 70 × weight^0.75`, times
a life-stage factor.

In [12]:
from datetime import date, timedelta
from typing import Literal
from langchain_core.tools import tool


LIFE_STAGE_FACTOR = {
    "kitten":           2.5,
    "neutered_adult":   1.2,
    "intact_adult":     1.4,
    "weight_loss":      0.8,
    "pregnant_nursing": 3.0,
    "senior":           1.1,
}


@tool
def daily_calorie_needs(
    weight_kg: float,
    life_stage: Literal["kitten", "neutered_adult", "intact_adult",
                        "weight_loss", "pregnant_nursing", "senior"],
) -> str:
    """Calculate a cat's daily energy requirement in kilocalories.

    weight_kg: current weight in kilograms.
    life_stage: which life-stage factor applies.
    """
    w = float(weight_kg)
    if w <= 0:
        return "Weight must be greater than zero."

    rer = 70 * (w ** 0.75)
    factor = LIFE_STAGE_FACTOR[life_stage]
    total = rer * factor

    return (f"weight {w:.2f} kg | RER {rer:.1f} kcal | "
            f"{life_stage} x{factor}\n"
            f"daily need: {total:.1f} kcal — about {total/3:.1f} per meal")


@tool
def cat_age_in_human_years(birth_date: str, today: str) -> str:
    """Convert a cat's real age into human-equivalent years.

    birth_date: YYYY-MM-DD.
    today: reference date, YYYY-MM-DD.
    """
    b = date.fromisoformat(birth_date)
    t = date.fromisoformat(today)
    days = (t - b).days
    if days < 0:
        return "Birth date is after the reference date."

    years = days / 365.25

    # first year counts as 15, second adds 9, each year after adds 4
    if years <= 1:
        human = 15 * years
    elif years <= 2:
        human = 15 + 9 * (years - 1)
    else:
        human = 24 + 4 * (years - 2)

    stage = "kitten" if years < 1 else "adult" if years < 11 else "senior"
    return (f"{years:.2f} real years ({days} days) — "
            f"about {human:.1f} human years, {stage}")


@tool
def next_vaccination_due(last_dose_date: str, vaccine: str, today: str) -> str:
    """Find when the next dose of a vaccine is due, renewed yearly.

    last_dose_date: YYYY-MM-DD.
    vaccine: 'core', 'rabies' or 'leukemia'.
    today: reference date, YYYY-MM-DD.
    """
    last = date.fromisoformat(last_dose_date)
    t = date.fromisoformat(today)
    due = last + timedelta(days=365)
    delta = (due - t).days

    if delta > 0:
        status = f"due in {delta} days"
    elif delta == 0:
        status = "due today"
    else:
        status = f"overdue by {abs(delta)} days"

    return f"{vaccine}: last {last}, next {due} — {status}"


@tool
def boarding_cost(
    nights: int,
    room: Literal["standard", "suite"],
    second_cat: Literal["yes", "no"] = "no",
    daily_play: Literal["yes", "no"] = "no",
) -> str:
    """Calculate total boarding cost in SAR.

    nights: number of nights.
    room: 'standard' (80/night) or 'suite' (140/night).
    second_cat: 'yes' if a second cat from the same home shares the room.
    daily_play: 'yes' to add the extra daily play session.
    """
    n = int(nights)
    if n < 1:
        return "Nights must be at least 1."

    rate = 80 if room == "standard" else 140
    total = rate * n
    lines = [f"{room}: {rate} x {n} = {total:.2f}"]

    if second_cat == "yes":
        total += rate * n / 2
        lines.append(f"second cat: +{rate * n / 2:.2f}")
    if daily_play == "yes":
        total += 40 * n
        lines.append(f"play sessions: +{40 * n:.2f}")
    if n >= 5:
        total -= total * 0.10
        lines.append("5+ nights: -10%")

    lines.append(f"total: {total:.2f} SAR")
    return "\n".join(lines)


@tool
def today() -> str:
    """Return today's date as YYYY-MM-DD.

    The model has no clock of its own, so anything involving an age or a due
    date has to start here.
    """
    return date.today().isoformat()


TOOLS = [daily_calorie_needs, cat_age_in_human_years,
         next_vaccination_due, boarding_cost, today]

print([t.name for t in TOOLS])

['daily_calorie_needs', 'cat_age_in_human_years', 'next_vaccination_due', 'boarding_cost', 'today']


In [13]:
print(daily_calorie_needs.invoke({"weight_kg": 4.0, "life_stage": "neutered_adult"}))
print()
print(daily_calorie_needs.invoke({"weight_kg": 8.0, "life_stage": "neutered_adult"}))
print()
print(daily_calorie_needs.invoke({"weight_kg": 4.0, "life_stage": "kitten"}))
print()
print(boarding_cost.invoke({"nights": 6, "room": "suite",
                            "second_cat": "yes", "daily_play": "yes"}))

weight 4.00 kg | RER 198.0 kcal | neutered_adult x1.2
daily need: 237.6 kcal — about 79.2 per meal

weight 8.00 kg | RER 333.0 kcal | neutered_adult x1.2
daily need: 399.6 kcal — about 133.2 per meal

weight 4.00 kg | RER 198.0 kcal | kitten x2.5
daily need: 495.0 kcal — about 165.0 per meal

suite: 140 x 6 = 840.00
second cat: +420.00
play sessions: +240.00
5+ nights: -10%
total: 1350.00 SAR


Doubling the weight raises the requirement by about 68%, not 100% — the
exponent is 0.75, not 1.

### Letting the model call them

One question that needs two different tools. The loop prints which tools were
called and with what arguments.

In [14]:
from langchain.agents import create_agent

cat_agent = create_agent(model=llm, tools=TOOLS, name="cat_calculator")

question = ("قطتي لولو وزنها 4.6 كيلو ومعقّمة وعمرها فوق السنتين. "
            "كم سعرة حرارية تحتاج في اليوم؟ وكم تكلفة إيواءها 6 ليالٍ "
            "في جناح واسع مع جلسة لعب يومية؟")

result = cat_agent.invoke({"messages": [{"role": "user", "content": question}]})

for m in result["messages"]:
    for tc in getattr(m, "tool_calls", []) or []:
        print(f"called {tc['name']} with {tc['args']}")

print()
print(result["messages"][-1].content)

called daily_calorie_needs with {'life_stage': 'neutered_adult', 'weight_kg': 4.6}
called boarding_cost with {'daily_play': 'yes', 'nights': 6, 'room': 'suite', 'second_cat': 'no'}

تحتاج قطتك لولو إلى 263.8 سعرة حرارية في اليوم. 
تكلفة إيواء قطتك لولو 6 ليالٍ في جناح واسع مع جلسة لعب يومية هي 972.00 ريال سعودي.


### Reading a cat's details out of free text

The profile is parsed into a Pydantic model rather than read out of prose,
since Part 2 writes these fields into long-term memory.

In [15]:
from pydantic import BaseModel, Field
from typing import Optional, Literal


class CatProfile(BaseModel):
    """Facts about a cat, pulled out of a free-text message."""
    name: Optional[str] = Field(default=None, description="cat's name if given")
    breed: Optional[str] = Field(default=None, description="breed if given")
    weight_kg: Optional[float] = Field(default=None, description="weight in kg")
    birth_date: Optional[str] = Field(default=None, description="YYYY-MM-DD")
    neutered: Literal["yes", "no", "unknown"] = Field(
        default="unknown",
        description="'yes' if neutered or spayed, 'no' if not, "
                    "'unknown' if the message does not say")
    last_vaccination: Optional[str] = Field(
        default=None, description="date of last vaccine, YYYY-MM-DD")


profile_extractor = llm.with_structured_output(CatProfile)

profile = profile_extractor.invoke(
    "عندي قطة اسمها لولو، شيرازية، وزنها 4.6 كيلو، مواليد 2023-04-12، "
    "معقّمة، وآخر تطعيم أخذته كان في 2025-09-20."
)

print(profile.model_dump_json(indent=2))

{
  "name": "لولو",
  "breed": "شيرازية",
  "weight_kg": 4.6,
  "birth_date": "2023-04-12",
  "neutered": "yes",
  "last_vaccination": "2025-09-20"
}


---

The two stores are in place and returning the right text. What follows uses
them: first a classifier that decides which one a question belongs to, then a
memory layer that keeps each cat's details between conversations.

---

# Part 2 — Routing and memory

## Picking a knowledge base

Every question goes to one store, the other, or both. The choice is made by
the model returning a constrained value, not by looking for keywords in the
text.

In [16]:
from typing import Literal
from pydantic import BaseModel, Field


class Route(BaseModel):
    """Which knowledge base can answer a question."""
    destination: Literal["care", "clinic", "both"] = Field(
        description=(
            "care   = feeding, water, litter, behaviour, grooming, breeds, "
            "symptoms and warning signs. "
            "clinic = prices, opening hours, booking, cancellation, the "
            "vaccination schedule, boarding terms, memberships. "
            "both   = answering properly needs care advice *and* clinic "
            "information together."
        )
    )
    reason: str = Field(description="One short sentence in English.")


router = llm.with_structured_output(Route)

ROUTER_PROMPT = """You direct questions from cat owners to one of two \
knowledge bases at Meow Clinic.

care   — the care guide: how to look after a cat, what behaviour means, \
which symptoms are worrying.
clinic — the services guide: what the clinic charges, when it opens, how \
booking works, boarding rules.

Judge what the owner actually needs to know, not which words they used. \
A question can mention a clinic service and still be a care question.

Question: {question}"""


def classify(question: str) -> Route:
    return router.invoke(ROUTER_PROMPT.format(question=question))


print(classify("كم سعر تنظيف الأسنان؟"))

destination='clinic' reason="The question is asking about the price of a service, which is related to the clinic's services guide."


The last question below is the interesting one. It contains the word
*تطعيم* (vaccination), which belongs to the clinic guide, but the owner is
asking whether a symptom is normal — a care question. Matching on keywords
would send it to the wrong store.

In [17]:
questions = [
    "قطتي تخربش الكنب، كيف أوقفها؟",
    "كم سعر تعقيم الأنثى وهل يشمل المتابعة؟",
    "قطتي شيرازية وفروها بدأ يتعقد، أمشطها بنفسي ولا أحجز لها عندكم؟",
    "قطتي بعد التطعيم صارت خاملة ليوم كامل، هل هذا طبيعي؟",
    "قطتي شيرازية، كم مرة أمشط فروها، وكم تكلفة الحلاقة الكاملة عندكم؟",
]

for q in questions:
    r = classify(q)
    print(f"{r.destination:6} | {q}")
    print(f"       | {r.reason}\n")

care   | قطتي تخربش الكنب، كيف أوقفها؟
       | The owner is asking about stopping their cat from scratching the sofa, which is a behavior issue.

clinic | كم سعر تعقيم الأنثى وهل يشمل المتابعة؟
       | The owner is asking about the price of a clinic service and what it includes.

care   | قطتي شيرازية وفروها بدأ يتعقد، أمشطها بنفسي ولا أحجز لها عندكم؟
       | The cat owner is asking about grooming their cat.

care   | قطتي بعد التطعيم صارت خاملة ليوم كامل، هل هذا طبيعي؟
       | The cat owner is asking about their cat's behavior after vaccination, which is a care-related concern.

both   | قطتي شيرازية، كم مرة أمشط فروها، وكم تكلفة الحلاقة الكاملة عندكم؟
       | The owner is asking about grooming their cat and the cost of a full shave at the clinic.



### Searching only what was chosen

The retriever for the other store is never called, so a wrong route produces a
visibly wrong answer rather than being quietly corrected.

In [18]:
def retrieve(question: str, destination: str):
    """Return (context_text, list_of_stores_actually_searched)."""
    if destination == "care":
        docs = care_retriever.invoke(question)
        return "\n\n".join(d.page_content for d in docs), ["care"]

    if destination == "clinic":
        docs = clinic_retriever.invoke(question)
        return "\n\n".join(d.page_content for d in docs), ["clinic"]

    care_docs = care_retriever.invoke(question)
    clinic_docs = clinic_retriever.invoke(question)
    text = ("=== دليل الرعاية ===\n"
            + "\n\n".join(d.page_content for d in care_docs)
            + "\n\n=== دليل الخدمات ===\n"
            + "\n\n".join(d.page_content for d in clinic_docs))
    return text, ["care", "clinic"]


ANSWER_PROMPT = """You answer cat owners at Meow Clinic, in Arabic.

Use only the context below. If it does not contain the answer, say so plainly \
instead of guessing. Never diagnose illness — if the context describes a \
warning sign, tell the owner to see a vet.

Context:
{context}

Question: {question}"""


def ask(question: str, verbose: bool = True):
    route = classify(question)
    context, searched = retrieve(question, route.destination)
    answer = llm.invoke(ANSWER_PROMPT.format(
        context=context, question=question)).content

    if verbose:
        print(f"route    : {route.destination} — {route.reason}")
        print(f"searched : {searched}")
        print(f"context  : {len(context)} characters\n")
        print(answer)
    return route, answer


ask("كم سعر تطعيم السعار، ومتى أجدده؟");

route    : clinic — The owner is asking about the price of a vaccination and when to renew it.
searched : ['clinic']
context  : 1018 characters

سعر تطعيم السعار هو مئة وثمانون ريالاً. يُجدد تطعيم السعار مرة واحدة كل اثني عشر شهراً.


In [19]:
print("=" * 70)
ask("قطتي ما أكلت من أمس، وش أسوي؟")
print("\n" + "=" * 70)
ask("قطتي شيرازية، كم مرة أمشط فروها، وكم تكلفة الحلاقة الكاملة عندكم؟");

route    : care — The cat owner is asking about their cat not eating, which is a care-related issue.
searched : ['care']
context  : 1218 characters

إذا لم تأكل قطتك لمدة 24 ساعة أو أكثر، فيجب عليك اصطحابها إلى الطبيب البيطري على الفور. هذا يمكن أن يكون علامة على مشكلة صحية تحتاج إلى عناية طبية فورية.

route    : both — The owner is asking about grooming their cat and the cost of a full shave at the clinic.
searched : ['care', 'clinic']
context  : 1976 characters

السلام عليكم، 
نظرًا لأن قطتك من سلالة الشيرازي، وهي سلالة طويلة الشعر، ينصح بتمشيط فروها يوميًا لمنع تكتلات مؤلمة تلتصق بالجلد.

أما بخصوص تكلفة الحلاقة الكاملة، فلم يُذكر ذلك في دليل الرعاية أو دليل الخدمات. يُرجى زيارة عيادة مواء للحصول على مزيد من المعلومات حول خدمات الحلاقة وتكلفتها.


### Routing as a handoff between sub-agents

`ask()` above routes to a retriever. The same decision can hand the question to
a **specialist agent** instead: each store gets its own agent, with its own
search tool and its own calculators, and the supervisor passes the question to
one of them.

The care agent cannot see the services guide and the clinic agent cannot see
the care guide — the separation is in the tools each agent holds, not in a
filter applied afterwards.

In [20]:
@tool
def search_care_guide(query: str) -> str:
    """Search the cat care guide: feeding, water, litter, behaviour, grooming,
    breed traits, and signs that a cat needs a vet.

    query: what to look for, in Arabic.
    """
    docs = care_retriever.invoke(query)
    return "\n\n".join(d.page_content for d in docs)


@tool
def search_clinic_services(query: str) -> str:
    """Search the Meow Clinic services guide: prices, opening hours, booking
    and cancellation rules, the vaccination schedule, boarding, memberships.

    query: what to look for, in Arabic.
    """
    docs = clinic_retriever.invoke(query)
    return "\n\n".join(d.page_content for d in docs)


CARE_BRIEF = ("أنتِ مختصة رعاية القطط في عيادة مواء. استخدمي أداة البحث في "
              "دليل الرعاية قبل أي إجابة، ولا تذكري أسعاراً أو إجراءات حجز "
              "فأنتِ لا تملكين هذه المعلومات. إذا كانت الحالة تستدعي طبيباً "
              "فقولي ذلك. أجيبي بالعربية.")

CLINIC_BRIEF = ("أنتِ موظفة خدمات في عيادة مواء. استخدمي أداة البحث في دليل "
                "الخدمات قبل أي إجابة، ولا تعطي نصائح رعاية أو تغذية فأنتِ لا "
                "تملكين هذه المعلومات. أجيبي بالعربية.")

care_agent = create_agent(
    model=llm,
    tools=[search_care_guide, daily_calorie_needs, cat_age_in_human_years, today],
    system_prompt=CARE_BRIEF,
    name="care_agent",
)

clinic_agent = create_agent(
    model=llm,
    tools=[search_clinic_services, boarding_cost, next_vaccination_due, today],
    system_prompt=CLINIC_BRIEF,
    name="clinic_agent",
)

SPECIALISTS = {"care": care_agent, "clinic": clinic_agent}

print("specialists:", list(SPECIALISTS))

specialists: ['care', 'clinic']


In [21]:
def run_specialist(name: str, question: str) -> dict:
    """Hand a question to one specialist and report what it did."""
    result = SPECIALISTS[name].invoke(
        {"messages": [{"role": "user", "content": question}]})
    calls = [tc["name"]
             for m in result["messages"]
             for tc in (getattr(m, "tool_calls", []) or [])]
    return {"agent": name, "tools": calls,
            "answer": result["messages"][-1].content}


def supervisor(question: str) -> dict:
    """Classify, then hand off. The supervisor never answers by itself."""
    route = classify(question)
    targets = ["care", "clinic"] if route.destination == "both" \
        else [route.destination]

    replies = [run_specialist(t, question) for t in targets]

    if len(replies) == 1:
        answer = replies[0]["answer"]
    else:
        joined = "\n\n".join(f"[{r['agent']}]\n{r['answer']}" for r in replies)
        answer = llm.invoke(
            "ادمجي هذين الردين في جواب عربي واحد متماسك دون إضافة أي معلومة "
            f"جديدة:\n\n{joined}").content

    return {"destination": route.destination, "reason": route.reason,
            "handoffs": [r["agent"] for r in replies],
            "tools_used": [t for r in replies for t in r["tools"]],
            "answer": answer}


for q in ["قطتي تخربش الكنب، كيف أوقفها؟",
          "كم سعر تنظيف الأسنان تحت التخدير؟",
          "قطتي شيرازية، كم مرة أمشط فروها، وكم تكلفة الحلاقة الكاملة عندكم؟"]:
    print("=" * 70)
    print(q)
    r = supervisor(q)
    print(f"route    : {r['destination']} — {r['reason']}")
    print(f"handed to: {r['handoffs']}")
    print(f"tools    : {r['tools_used']}")
    print()
    print(r["answer"][:350])
    print()

قطتي تخربش الكنب، كيف أوقفها؟
route    : care — The cat owner is asking about stopping their cat from scratching the sofa, which is a behavior issue.
handed to: ['care']
tools    : ['search_care_guide']

الخربشة سلوك طبيعي لشحذ الأظافر وترك علامة رائحة، ولا يمكن منعها بالعقاب. ضع عمود خربشة طويلاً بما يكفي لتمدد القطة كاملة، وثبته بقرب المكان الذي تخربشه حالياً. قص أطراف الأظافر كل أسبوعين إلى ثلاثة أسابيع يقلل الضرر.

كم سعر تنظيف الأسنان تحت التخدير؟
route    : clinic — The owner is asking about the price of a specific clinic service, teeth cleaning under anesthesia.
handed to: ['clinic']
tools    : ['search_clinic_services']

سعر تنظيف الأسنان تحت التخدير ثمانمئة ريال ويشمل الفحص قبل التخدير.

قطتي شيرازية، كم مرة أمشط فروها، وكم تكلفة الحلاقة الكاملة عندكم؟
route    : both — The owner is asking about grooming their cat and the cost of a full shave at the clinic.
handed to: ['care', 'clinic']
tools    : ['search_care_guide', 'search_care_guide', 'search_clinic_services', 'search_cli

Each run names the agent that received the question and the tools that agent
called. `search_care_guide` never appears under `clinic_agent`, and
`search_clinic_services` never appears under `care_agent`, because neither
agent holds the other's tool.

The functional pipeline in Part 3 reuses the same classifier, calling the
retrievers directly rather than through an agent, so that retries and the
approval step sit at the level of individual steps.

## Memory

Two different things, kept apart:

- **Short-term** — a checkpointer holding the messages of one conversation,
  keyed by `thread_id`. It disappears when the owner starts a new chat.
- **Long-term** — a Store holding facts about each cat, keyed by owner and cat
  name. It is not attached to any conversation at all.

### Short-term: one conversation remembering itself

The second message says *"وكم عمرها"* without repeating the cat's name or
birth date. The agent can only answer it by reading the earlier turn out of
the checkpointer.

In [22]:
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()
chat_agent = create_agent(
    model=llm, tools=TOOLS, name="cat_assistant", checkpointer=checkpointer)

thread_a = {"configurable": {"thread_id": "chat-A"}}

r1 = chat_agent.invoke({"messages": [{"role": "user", "content":
    "قطتي اسمها لولو، مواليد 2023-04-12، ووزنها 4.6 كيلو ومعقّمة."}]},
    thread_a)
print("turn 1:", r1["messages"][-1].content[:200])

r2 = chat_agent.invoke({"messages": [{"role": "user", "content":
    "وكم عمرها بالسنين البشرية؟ اليوم 2026-08-10"}]}, thread_a)
print("\nturn 2:", r2["messages"][-1].content[:300])

print(f"\nmessages kept in thread chat-A: {len(r2['messages'])}")

turn 1: لولو تحتاج إلى 263.8 كيلو سعرة حرارية يومياً، وهي تعادل 87.9 سعرة حرارية لكل وجبة. عمر لولو الحقيقي هو 1 سنة و 0 أشهر و 0 أيام، وهو يعادل 15 سنة بشرية.

turn 2: عمر لولو الحقيقي هو 3 سنوات و 4 أشهر و 0 أيام، وهو يعادل 29.3 سنة بشرية.

messages kept in thread chat-A: 10


### Long-term: a Store, separate from any conversation

The profile extracted in Part 1 is written under a namespace of owner and cat
name.

In [23]:
from langgraph.store.memory import InMemoryStore

store = InMemoryStore()
OWNER = "owner-001"
NS = ("cats", OWNER)


def save_cat(profile: CatProfile) -> None:
    store.put(NS, profile.name, profile.model_dump())


def load_cat(name: str) -> dict | None:
    item = store.get(NS, name)
    return item.value if item else None


# profile came from the structured extraction at the end of Part 1
save_cat(profile)

# a second cat, so the lookup has to pick the right one
save_cat(profile_extractor.invoke(
    "قطة ثانية اسمها بسبوسة، سيامي، وزنها 3.2 كيلو، مواليد 2024-11-02، "
    "غير معقّمة، آخر تطعيم 2026-01-15."))

print("stored cats:", [it.key for it in store.search(NS)])
print()
print(load_cat("لولو"))

stored cats: ['لولو', 'بسبوسة']

{'name': 'لولو', 'breed': 'شيرازية', 'weight_kg': 4.6, 'birth_date': '2023-04-12', 'neutered': 'yes', 'last_vaccination': '2025-09-20'}


### The cross-thread test

A brand new conversation, `chat-B`, with its own empty checkpointer history.
It has never been told anything about لولو.

First without the Store — the agent has nothing to work from. Then with the
profile loaded out of the Store and handed to it.

In [24]:
thread_b = {"configurable": {"thread_id": "chat-B"}}

# new thread, nothing in its history and no store lookup
blind = chat_agent.invoke({"messages": [{"role": "user", "content":
    "كم سعرة حرارية تحتاج لولو في اليوم؟"}]}, thread_b)

print("--- chat-B, no store lookup ---")
print(f"messages in this thread: {len(blind['messages'])}\n")

for m in blind["messages"]:
    for tc in getattr(m, "tool_calls", []) or []:
        print(f"called {tc['name']} with {tc['args']}")

print()
print(blind["messages"][-1].content[:250])

--- chat-B, no store lookup ---
messages in this thread: 4

called daily_calorie_needs with {'life_stage': 'kitten', 'weight_kg': 5}

تحتاج لولو إلى 585.1 سعرة حرارية في اليوم.


In [25]:
# 2 · same new thread, this time the facts are fetched from the Store
thread_c = {"configurable": {"thread_id": "chat-C"}}

facts = load_cat("لولو")
print("loaded from Store:", facts, "\n")

stage = "neutered_adult" if facts["neutered"] == "yes" else "intact_adult"
prompt = (f"قطة اسمها {facts['name']}، وزنها {facts['weight_kg']} كيلو، "
          f"حالتها {stage}. كم سعرة حرارية تحتاج في اليوم؟")

informed = chat_agent.invoke(
    {"messages": [{"role": "user", "content": prompt}]}, thread_c)

for m in informed["messages"]:
    for tc in getattr(m, "tool_calls", []) or []:
        print(f"called {tc['name']} with {tc['args']}")

print()
print(informed["messages"][-1].content[:300])

loaded from Store: {'name': 'لولو', 'breed': 'شيرازية', 'weight_kg': 4.6, 'birth_date': '2023-04-12', 'neutered': 'yes', 'last_vaccination': '2025-09-20'} 

called daily_calorie_needs with {'life_stage': 'neutered_adult', 'weight_kg': 4.6}

تحتاج القطة لولو إلى 263.8 سعرة حرارية في اليوم، ويمكن تقسيمها إلى ثلاث وجبات، كل وجبة تحتوي على 87.9 سعرة حرارية.


Compare the two `weight_kg` values. In `chat-B` the agent had no record of
لولو, so rather than saying it did not know, it invented a weight and answered
confidently. In `chat-C` the same agent received `4.6` — a number that was
typed into `chat-A` and reached `chat-C` only through the Store, since the two
threads share no history at all.

---

Routing and memory both work through ordinary function calls. The next section
rebuilds the same flow with `@task` and `@entrypoint`, which is what makes
retries and a mid-run pause possible.


---

# Part 3 — Functional API, failure handling, and human approval

The flow so far is a chain of plain function calls. Rebuilding it with
`@task` and `@entrypoint` gives each step its own retry behaviour, and lets
the run pause in the middle and resume later — which is what booking an
appointment needs.

### Checking the retry argument name

`task` has taken the retry policy under different keyword names across
LangGraph versions, so we read the signature instead of assuming.

In [26]:
import inspect
from langgraph.func import task, entrypoint
from langgraph.types import RetryPolicy, interrupt, Command

params = inspect.signature(task).parameters
print("task() accepts:", list(params))

RETRY_KW = "retry_policy" if "retry_policy" in params else "retry"
print("using keyword:", RETRY_KW)

RETRY = {RETRY_KW: RetryPolicy(max_attempts=3)}

task() accepts: ['__func_or_none__', 'name', 'retry_policy', 'cache_policy', 'timeout', 'kwargs']
using keyword: retry_policy


### Strategy 1 — retry a transient failure

A real `RetryPolicy`, not a loop with `sleep`. The task below fails its first
two attempts on purpose and prints each one, so the retries are visible in the
output rather than merely claimed.

In [27]:
attempts = {"n": 0}


@task(**RETRY)
def flaky_price_lookup(service: str) -> str:
    """Stands in for a call that fails intermittently."""
    attempts["n"] += 1
    print(f"  attempt {attempts['n']} for {service!r}")
    if attempts["n"] < 3:
        raise ConnectionError("price service temporarily unavailable")
    return f"{service}: 110 SAR"


@entrypoint()
def retry_demo(service: str) -> str:
    return flaky_price_lookup(service).result()


print(retry_demo.invoke("استحمام وتجفيف"))
print(f"total attempts: {attempts['n']}")

  attempt 1 for 'استحمام وتجفيف'
  attempt 2 for 'استحمام وتجفيف'
  attempt 3 for 'استحمام وتجفيف'
استحمام وتجفيف: 110 SAR
total attempts: 3


### Strategy 2 — fall back when the classifier is unavailable

If routing fails there is still a sensible answer available: search both
stores. Slower and less precise, but the owner gets a reply.

In [28]:
@task(**RETRY)
def classify_task(question: str, force_failure: bool = False) -> Route:
    if force_failure:
        raise RuntimeError("routing model unavailable")
    return classify(question)


def classify_or_both(question: str, force_failure: bool = False) -> Route:
    try:
        return classify_task(question, force_failure).result()
    except Exception as e:
        print(f"  routing failed ({type(e).__name__}) — searching both stores")
        return Route(destination="both",
                     reason="fallback after the classifier failed")


@entrypoint()
def routing_demo(payload: dict) -> dict:
    r = classify_or_both(payload["question"], payload.get("break_it", False))
    return {"destination": r.destination, "reason": r.reason}


print("normal:")
print(" ", routing_demo.invoke({"question": "كم سعر تطعيم السعار؟"}))

print("\nwith the classifier broken:")
print(" ", routing_demo.invoke(
    {"question": "كم سعر تطعيم السعار؟", "break_it": True}))

normal:
  {'destination': 'clinic', 'reason': 'The owner is asking about the price of a vaccination service.'}

with the classifier broken:
  routing failed (RuntimeError) — searching both stores
  {'destination': 'both', 'reason': 'fallback after the classifier failed'}


### Strategy 3 — refuse to answer past the model's remit

Some questions are not answerable by a chatbot at all. A separate triage step
reads the question and decides whether the cat needs a vet, and that decision
is a constrained field rather than something parsed out of prose.

`needs_vet` is `"yes"`/`"no"` rather than a boolean. Groq validates the tool
call against the schema on its own side and rejects the whole request if the
model writes `"false"` as a string, which it does often enough to matter — so
the schema asks for what the model reliably produces.

In [29]:
class Triage(BaseModel):
    """Whether a described situation needs a veterinarian rather than advice."""
    needs_vet: Literal["yes", "no"] = Field(
        description="'yes' when the cat should be examined by a vet rather "
                    "than advised about, 'no' otherwise")
    urgency: Literal["routine", "soon", "emergency"] = Field(
        description="emergency for anything that cannot wait until tomorrow")
    concern: str = Field(description="one short sentence in English")


triage_llm = llm.with_structured_output(Triage)

TRIAGE_PROMPT = """A cat owner has written to Meow Clinic. Decide whether \
this needs a veterinarian to examine the cat.

Treat as an emergency: not eating for a day or more, straining to urinate, \
open-mouth breathing, repeated vomiting, collapse, bleeding, a swollen \
abdomen, or sudden inability to stand.

Message: {question}"""


@task(**RETRY)
def triage_task(question: str) -> Triage:
    return triage_llm.invoke(TRIAGE_PROMPT.format(question=question))


@entrypoint()
def triage_demo(question: str) -> dict:
    t = triage_task(question).result()
    return t.model_dump()


for q in ["كم مرة أمشط فرو قطتي الشيرازية؟",
          "قطتي تحاول تتبول من ساعتين وما ينزل شيء"]:
    print(q)
    print(" ", triage_demo.invoke(q), "\n")

كم مرة أمشط فرو قطتي الشيرازية؟
  {'needs_vet': 'no', 'urgency': 'routine', 'concern': 'how often to brush a shirazi cat'} 

قطتي تحاول تتبول من ساعتين وما ينزل شيء
  {'needs_vet': 'yes', 'urgency': 'emergency', 'concern': 'cat trying to urinate for two hours but nothing comes out'} 



## The full assistant

All of it in one `@entrypoint`: triage, routing with its fallback, retrieval
from whichever store was chosen, and the answer. When the owner asks to book,
the run stops at `interrupt()` and waits.

Booking is where stopping matters — it takes a 50 SAR deposit and a slot in
the diary, and cancelling inside 24 hours forfeits the deposit.

In [30]:
@task
def retrieve_task(question: str, destination: str) -> dict:
    context, searched = retrieve(question, destination)
    return {"context": context, "searched": searched}


@task(**RETRY)
def answer_task(question: str, context: str) -> str:
    return llm.invoke(ANSWER_PROMPT.format(
        context=context, question=question)).content


@task
def confirm_booking(service: str, slot: str) -> str:
    """Stands in for writing to the clinic diary."""
    return f"BOOKED — {service} at {slot}, 50 SAR deposit taken."


@entrypoint(checkpointer=InMemorySaver())
def assistant(payload: dict) -> dict:
    question = payload["question"]
    booking = payload.get("booking")     # {"service": ..., "slot": ...} or None

    t = triage_task(question).result()
    route = classify_or_both(question)
    r = retrieve_task(question, route.destination).result()
    answer = answer_task(question, r["context"]).result()

    if t.needs_vet == "yes":
        answer = ("⚠️ هذي الحالة تحتاج عرض القطة على طبيب.\n\n" + answer)

    booked = None
    if booking:
        decision = interrupt({
            "needs": "staff approval before the diary is written to",
            "service": booking["service"],
            "slot": booking["slot"],
            "deposit": "50 SAR",
            "triage": t.urgency,
        })
        if decision == "approve":
            booked = confirm_booking(booking["service"],
                                     booking["slot"]).result()
        else:
            booked = f"NOT BOOKED — {decision}"

    return {
        "destination": route.destination,
        "searched": r["searched"],
        "urgency": t.urgency,
        "needs_vet": t.needs_vet,
        "answer": answer,
        "booking": booked,
    }


print("built")

built


### A run with no booking

In [31]:
cfg1 = {"configurable": {"thread_id": "run-1"}}

out = assistant.invoke(
    {"question": "كم سعر تقليم الأظافر عندكم؟"}, cfg1)

for k, v in out.items():
    print(f"{k}: {v}")

destination: clinic
searched: ['clinic']
urgency: routine
needs_vet: no
answer: سعر تقليم الأظافر عندنا أربعون ريالاً.
booking: None


### A run that pauses

The reply comes back with `__interrupt__` instead of a result. Nothing has
been written to the diary yet.

In [32]:
cfg2 = {"configurable": {"thread_id": "run-2"}}

paused = assistant.invoke({
    "question": "أبي أحجز استحمام وتجفيف لقطتي لولو",
    "booking": {"service": "استحمام وتجفيف", "slot": "الأحد 4:00 م"},
}, cfg2)

pending = paused["__interrupt__"][0].value
print("PAUSED — waiting for staff")
for k, v in pending.items():
    print(f"  {k}: {v}")

PAUSED — waiting for staff
  needs: staff approval before the diary is written to
  service: استحمام وتجفيف
  slot: الأحد 4:00 م
  deposit: 50 SAR
  triage: routine


### Resuming it

`Command(resume="approve")` sends the staff decision back into the paused run,
which picks up from the `interrupt()` and finishes.

In [33]:
done = assistant.invoke(Command(resume="approve"), cfg2)

print("booking:", done["booking"])
print("route  :", done["destination"], done["searched"])
print()
print(done["answer"][:250])

booking: BOOKED — استحمام وتجفيف at الأحد 4:00 م, 50 SAR deposit taken.
route  : clinic ['clinic']

يمكنك حجز استحمام وتجفيف لقطتك لولو. سعر الاستحمام والتجفيف هو مئة وعشرة ريالات. يمكنك الحجز عبر تطبيق العيادة أو بالاتصال. يُطلب مبلغ تأكيد قدره خمسون ريالاً لحجز الموعد، ويُخصم من قيمة الزيارة.


### And a run that is declined

Same pause, different decision — the run finishes without a booking.

In [34]:
cfg3 = {"configurable": {"thread_id": "run-3"}}

assistant.invoke({
    "question": "أبي أحجز تنظيف أسنان لبسبوسة",
    "booking": {"service": "تنظيف الأسنان", "slot": "الثلاثاء 11:00 ص"},
}, cfg3)

refused = assistant.invoke(
    Command(resume="الموعد محجوز لقطة أخرى، نعرض الأربعاء 10 ص بدلاً منه"), cfg3)

print(refused["booking"])

NOT BOOKED — الموعد محجوز لقطة أخرى، نعرض الأربعاء 10 ص بدلاً منه


### An emergency asking to book

Triage marks it, the warning goes in front of the answer, and the staff member
sees the urgency in the pause before deciding.

In [35]:
cfg4 = {"configurable": {"thread_id": "run-4"}}

urgent = assistant.invoke({
    "question": "قطتي تحاول تتبول وما ينزل شيء من الصباح، أبي موعد",
    "booking": {"service": "كشف طوارئ", "slot": "اليوم 8:00 م"},
}, cfg4)

pending = urgent["__interrupt__"][0].value
print("triage in the pause:", pending["triage"], "\n")

final = assistant.invoke(Command(resume="approve"), cfg4)
print("needs_vet:", final["needs_vet"], "|", final["urgency"])
print("booking  :", final["booking"])
print()
print(final["answer"][:300])

triage in the pause: emergency 

needs_vet: yes | emergency
booking  : BOOKED — كشف طوارئ at اليوم 8:00 م, 50 SAR deposit taken.

⚠️ هذي الحالة تحتاج عرض القطة على طبيب.

يجب عليك اصطحاب قطتك إلى طبيب بيطري على الفور، حيث أن محاولة التبول بلا نتيجة أو التبول المتكرر بكميات قليلة هي حالة طارئة في الذكور تحديداً.


---

The pipeline answers, retries, and stops for approval. What remains is a check
on the answers themselves, and the tracing that shows where the time goes.

---

# Part 4 — Reviewing answers, and tracing

## Workflow pattern: evaluator-optimizer

Of the five workflow patterns, this assistant uses **evaluator-optimizer** for
its final answer: one model call writes the reply, a second one judges it
against the retrieved context, and a failed judgement sends it back to be
rewritten.

It fits because the failure this assistant is most prone to is a fluent
sentence about a service or a price that no retrieved chunk supports — the
grooming answer earlier claimed the clinic offers a booking without ever
having searched the services guide. A reviewer that sees the answer and the
context together catches that; nothing earlier in the pipeline can.

In [36]:
class Grounding(BaseModel):
    """Whether every claim in an answer is supported by the retrieved text."""
    grounded: Literal["yes", "no"] = Field(
        description="'no' if the answer states any fact, price or service "
                    "that does not appear in the context")
    problem: str = Field(
        description="the unsupported claim, or 'none'. One short sentence.")


grounding_llm = llm.with_structured_output(Grounding)

REVIEW_PROMPT = """You check a support reply against the source text it was \
supposed to come from.

Mark it 'no' if it states any price, service, duration or rule that is not in \
the context. Advice to see a vet, and admitting something was not found, are \
always acceptable.

Context:
{context}

Reply:
{answer}"""

REVISE_PROMPT = """Rewrite this reply so every claim comes from the context. \
Drop anything unsupported instead of softening it. If the context does not \
answer the question, say so plainly. Reply in Arabic.

Context:
{context}

Question: {question}

Previous reply: {answer}
Problem found: {problem}"""


@task(**RETRY)
def review_answer(answer: str, context: str) -> Grounding:
    return grounding_llm.invoke(REVIEW_PROMPT.format(
        context=context, answer=answer))


@task(**RETRY)
def revise_answer(question: str, context: str,
                  answer: str, problem: str) -> str:
    return llm.invoke(REVISE_PROMPT.format(
        context=context, question=question,
        answer=answer, problem=problem)).content


print("built")

built


In [37]:
MAX_ROUNDS = 3


@entrypoint()
def answer_reviewed(question: str) -> dict:
    route = classify_or_both(question)
    r = retrieve_task(question, route.destination).result()
    answer = answer_task(question, r["context"]).result()

    history = []
    for round_no in range(1, MAX_ROUNDS + 1):
        verdict = review_answer(answer, r["context"]).result()
        history.append({"round": round_no,
                        "grounded": verdict.grounded,
                        "problem": verdict.problem})
        if verdict.grounded == "yes":
            break
        answer = revise_answer(question, r["context"],
                               answer, verdict.problem).result()

    return {"destination": route.destination,
            "searched": r["searched"],
            "rounds": history,
            "answer": answer}


for q in ["كم سعر تطعيم السعار؟",
          "قطتي شيرازية وفروها بدأ يتعقد، أمشطها بنفسي ولا أحجز لها عندكم؟"]:
    print("=" * 70)
    print(q, "\n")
    res = answer_reviewed.invoke(q)
    print("route:", res["destination"], res["searched"])
    for h in res["rounds"]:
        print(f"  round {h['round']}: grounded={h['grounded']} — {h['problem']}")
    print("\n" + res["answer"][:400])

كم سعر تطعيم السعار؟ 

route: clinic ['clinic']
  round 1: grounded=yes — none

سعر تطعيم السعار هو مئة وثمانون ريالاً.
قطتي شيرازية وفروها بدأ يتعقد، أمشطها بنفسي ولا أحجز لها عندكم؟ 

route: care ['care']
  round 1: grounded=no — حجز موعد لتمشيط فروها في عيادة مواء
  round 2: grounded=yes — none

نعم، يمكنك تمشيط فرو قطتك الشيرازية بنفسك. السلالات طويلة الشعر مثل الشيرازي تحتاج إلى تمشيط يومياً لمنع تكتلات مؤلمة تلتصق بالجلد.


## LangSmith

The environment variable is `LANGCHAIN_TRACING_V2`. `LANGSMITH_TRACING_V2`
looks plausible and does nothing at all — no error, no trace — so the
connection is verified below rather than assumed.

In [38]:
%pip install -qU langsmith

In [39]:
from langsmith import Client

client = Client()

# Tracing was switched on in the setup cell at the top, before langchain was
# imported. This is a real request, so a bad key fails here and not silently.
try:
    list(client.list_projects(limit=1))
    print("connected to LangSmith")
    print("project:", os.environ["LANGCHAIN_PROJECT"])
except Exception as e:
    print("NOT connected:", type(e).__name__, str(e)[:200])

connected to LangSmith
project: meow-clinic-capstone


### A traced run

In [40]:
from langchain_core.tracers.context import tracing_v2_enabled

with tracing_v2_enabled(project_name=os.environ["LANGCHAIN_PROJECT"]):
    traced = answer_reviewed.invoke(
        "كم تكلفة إيواء قطتين خمس ليالٍ في غرفة عادية؟")

print("route:", traced["destination"], traced["searched"])
for h in traced["rounds"]:
    print(f"  round {h['round']}: grounded={h['grounded']}")
print()
print(traced["answer"][:300])

route: clinic ['clinic']
  round 1: grounded=yes

تكلفة إيواء قطة واحدة في الغرفة العادية هي 80 ريالاً في الليلة. إذا أردنا إيواء قطتين في نفس الغرفة، فإن القطة الثانية ستكون بنصف السعر، أي 40 ريالاً في الليلة.

إذا أردنا إيواء القطتين لخمس ليالٍ، فإن التكلفة الإجمالية ستكون:
- القطة الأولى: 80 ريالاً * 5 ليال = 400 ريال
- القطة الثانية: 40 ريالاً 


### Reading the trace back

Traces are uploaded in the background, so the run list is fetched after a
short pause. Each row is one step of the run with the time it took.

In [41]:
import time

# traces upload in the background; flush and wait before reading them back
try:
    client.flush()
except Exception:
    pass
time.sleep(15)

project = os.environ["LANGCHAIN_PROJECT"]

try:
    runs = list(client.list_runs(project_name=project, limit=25))
except Exception as e:
    print(f"{type(e).__name__}: {e}")
    print("\nThe project appears once the first trace arrives. If this keeps "
          "failing, restart the session and run from the top — tracing must be "
          "on before langchain is imported.")
    runs = []

if runs:
    print(f"{len(runs)} runs in {project}\n")
    print(f"{'name':32} {'type':12} {'seconds':>8}")
    print("-" * 54)
    for r in runs:
        secs = ((r.end_time - r.start_time).total_seconds()
                if r.end_time and r.start_time else None)
        shown = f"{secs:8.2f}" if secs is not None else "       -"
        print(f"{r.name[:31]:32} {r.run_type:12} {shown}")

/tmp/ipykernel_3481/617683311.py:13: DeprecationWarning: list_runs() is deprecated and will be removed after Jan 31, 2027. Use client.runs.query() instead. See https://docs.langchain.com/langsmith/smithdb-sdk-migration#runs-query for the migration guide.
  runs = list(client.list_runs(project_name=project, limit=25))


25 runs in meow-clinic-capstone

name                             type          seconds
------------------------------------------------------
PydanticToolsParser              parser           0.00
ChatGroq                         llm              4.23
RunnableSequence                 chain            4.23
review_answer                    chain            4.24
ChatGroq                         llm              0.91
answer_task                      chain            0.91
VectorStoreRetriever             retriever        0.05
retrieve_task                    chain            0.05
PydanticToolsParser              parser           0.00
ChatGroq                         llm              0.30
RunnableSequence                 chain            0.30
classify_task                    chain            0.30
answer_reviewed                  chain            5.50
LangGraph                        chain            5.51
PydanticToolsParser              parser           0.00
ChatGroq                        

### What the trace showed

Tracing was on from the first cell, so the table covers every model call in the
notebook rather than only the last run.

One complete question — `answer_reviewed` — took 5.50 s end to end, and it
breaks down badly unevenly:

| step | seconds |
|---|---|
| `retrieve_task` | 0.05 |
| `classify_task` | 0.30 |
| `answer_task` | 0.91 |
| `review_answer` | 4.24 |

Retrieval is under 1% of the run. Routing is cheap because the classifier
returns one constrained field. Almost everything else is the reviewer, which
costs roughly four and a half times what writing the answer costs — it receives
the draft *and* the full retrieved context, so it reads far more than the
writer does.

On the grooming question, which failed its first review, the loop added
`review 4.25 → revise 3.39 → review 4.24`, about 12 s to remove one unsupported
clause from an answer that was already written.

That trade is worth making when a reply states a price or names a service the
clinic may not offer, and hard to justify on a question like "how often should
I brush her". The change it points to is running the review conditionally —
only when the draft contains a number or a service name — rather than on every
question.

---

## Summary

Everything above ran in one pass, top to bottom, with outputs saved.

| | |
|---|---|
| Tools | 7, all computing from their arguments |
| Specialists | `care_agent`, `clinic_agent` — each holding only its own search tool |
| Routing | `care` / `clinic` / `both`, all three exercised |
| Retrieval | 7 of 8 grounding checks pass; the failure is diagnosed above |
| Memory | short-term per thread, long-term Store, proven across three threads |
| Human approval | 3 pause-and-resume cycles: approved, refused, and an emergency |
| Error handling | retry, routing fallback, and escalation to a vet |
| Pattern | evaluator-optimizer — caught and removed an unsupported claim |
| Tracing | 25 runs in LangSmith with per-step timings |

The full write-up, one section at a time, is in
[`docs/writeup.md`](docs/writeup.md).